# Does Text Length Explain the Clustering?
### Testing unsupervised subject discovery on early modern normative texts from Bern (1528–1798)

**Notebook by:** Kamyab Karimi and C. Annemieke Romein (both University of Twente, Enschede, the Netherlands)  
**Workshop:** HISTMET & Annif, Part One: HISTMET · [Part Two: Annif](https://colab.research.google.com/github/CARomein/WorkshopBern2026_HISTMET_ANNIF/blob/master/02_Annif_subject_indexing.ipynb)  
**Version:** 1.0 (September 2026) · **DOI:** *to be added upon deposit in Zenodo* · **Licence:** *to be added*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CARomein/WorkshopBern2026_HISTMET_ANNIF/blob/master/01_HISTMET_text_length.ipynb)

## About this notebook

This notebook works entirely with early modern normative texts from Bern (*Policeyordnungen*, the regulatory ordinances or *mandates* by which early modern authorities governed public order, economic life and social conduct). The question it asks, however, comes from elsewhere. The project HISTMET (HIStorical Themes via METadata), funded by the Dutch Research Council (NWO), investigated whether subject metadata can be derived from historical texts *without* labelling them by hand. Its corpus, early modern Dutch provincial resolutions, was cut into short text regions (on average some 178 characters, roughly two lines), which were converted into numerical representations with a multilingual language model and grouped automatically. The resulting clusters were coherent, but they sorted by documentary genre rather than by theme: petitions ended up with petitions and resolutions with resolutions, irrespective of what they were about. The project's explanation is that the length of the text regions, rather than the early modern language, governs what the model picks up.

That explanation cannot be tested on the Dutch material, since it has no subject labels. The Bernese mandates can serve as a test case: they are about ten times longer, and each text has been assigned subjects by hand. In this notebook we run the same method twice on the same documents, once on the whole texts and once on a single 178-character piece from each. Documents, method and number of texts stay the same; only the amount of text changes.

This notebook is a simplified teaching version of the external validation reported in the HISTMET article (Section 5.4). The article's design differs in two respects, explained in Steps 5 and 6, so the scores obtained here are not identical to the published ones; the direction of the result is.

### The data

The file `bern_prepared.csv.gz` contains 4,928 of the 4,930 Bernese texts used in Part Two (two texts have no usable transcription). It uses the 2024 transcription (Transkribus Text Titan I, character error rate c. 2.95%), with words split across lines rejoined and layout markup removed. Each text carries one subject from Level 3 of the hierarchical SKOS vocabulary derived from the *Repertorium der Policeyordnungen* (25 categories, e.g. *4.6 Trade and Services*), as selected in C. Annemieke Romein's earlier project *A Game of Thrones* from the labels assigned by hand; many texts carry more than one label in the original data.

### How to use this notebook

Run the cells from top to bottom, either with the ▶ button or with `Shift + Enter`; the whole notebook takes about five minutes and installs nothing on your own computer. It requires a GPU: check that *Runtime → Change runtime type* is set to **T4 GPU** (without it, Step 5 takes roughly fifteen minutes). Lines beginning with `!` are command-line instructions rather than Python. Note that a notebook opened from GitHub is not saved anywhere: choose *File → Save a copy in Drive* at the start if you wish to keep your edits and results. Colab deletes all files when the session ends, so download anything you want to keep.

| Step | What happens |
|---|---|
| 1. Install | Install the language-model and clustering libraries |
| 2. Data | Download the Bernese texts |
| 3. Settings | Set parameters and define helper functions |
| 4. Two versions | Create the whole-text and short-piece versions |
| 5. Encoding | Turn each text into a vector with a language model |
| 6. Clustering | Group the vectors and measure agreement with the subjects |
| 7. Result | Compare both versions in a bar chart |
| 8. Map | Show the same comparison as a two-dimensional map |

# **1. INSTALL**

This cell installs three libraries that are not available in Colab by default: `sentence-transformers` (to load the language model), `umap-learn` (to reduce the dimensionality of the data) and `hdbscan` (to cluster it).

In [ ]:
#Step 1. Install
!pip install -q sentence-transformers umap-learn hdbscan
print("done")

# **2. DATA**

This cell downloads `bern_prepared.csv.gz` from the workshop repository. Should the download fail, empty `DATA_URL` (i.e. set it to `""`) and run the cell again; you will then be asked to select the file from your own computer. The cell ends by listing the columns: `LawID` (the identifier of the text), `gold` and `gold_name` (the hand-assigned subject as code and name) and `text` (the transcription).

In [ ]:
#Step 2. Load the data
DATA_URL = "https://raw.githubusercontent.com/CARomein/WorkshopBern2026_HISTMET_ANNIF/master/histmet/bern_prepared.csv.gz"   # set to "" to upload the file yourself

import os
import urllib.request
import pandas as pd

FNAME = "bern_prepared.csv.gz"

if not os.path.exists(FNAME):
    if DATA_URL:
        print("downloading")
        urllib.request.urlretrieve(DATA_URL, FNAME)
    else:
        try:
            from google.colab import files
            print(f"Choose {FNAME} from your computer.")
            files.upload()
        except ImportError:
            raise FileNotFoundError(f"{FNAME} not found. Put it next to this notebook.")

if not os.path.exists(FNAME):
    raise FileNotFoundError(f"{FNAME} still not here. Re-run and pick the file.")

cols = list(pd.read_csv(FNAME, nrows=0).columns)
print(f"{FNAME}, {os.path.getsize(FNAME)/1e6:.1f} MB")
print("columns:", cols)

if "gold_name" not in cols:
    print("")
    print("Older copy. For readable subject names, run os.remove(FNAME) in a new")
    print("cell, then run this cell again with the current file.")

# **3. SETTINGS**

This cell fixes the parameters of the experiment and defines a few helper functions; it does not yet compute anything. `CHUNK_CHARS` sets the length of the short pieces (178 characters, the average length of the HISTMET regions), and `SEED` makes the random choices reproducible. The UMAP and HDBSCAN parameters govern the clustering (Step 6). The two `HISMET_…_AMI` values are the scores obtained on the Dutch HISTMET regions, measured against the project's ten lexicon-based themes and against documentary form respectively; they serve as a point of reference in Step 7.

Agreement is measured with **adjusted mutual information** (AMI): 0 means the clusters are no better than chance at reproducing the hand-assigned subjects, 1 means they reproduce them perfectly. The function `biggest_group_share` guards against a common failure: if one cluster swallows most documents, the AMI is meaningless.

In [ ]:
#Step 3. Settings
import re
import warnings
import numpy as np
from sklearn.metrics import adjusted_mutual_info_score

warnings.filterwarnings("ignore", message=".*n_jobs value.*")

CHUNK_CHARS = 178      # our own region length
MIN_CHARS   = 30
SEED        = 20260911

MIN_CLUSTER_SIZE = 15
MIN_SAMPLES      = 5
SELECTION        = "leaf"
UMAP_DIM         = 50
UMAP_NEIGHBOURS  = 15

HISMET_SUBJECT_AMI = 0.1054   # our clusters against subject
HISMET_FORM_AMI    = 0.1637   # our clusters against document type

def tidy(t):
    return re.sub(r"\s+", " ", str(t)).strip()

def cut_into_pieces(text, target=CHUNK_CHARS):
    """Cut into pieces of roughly `target` characters, never mid-word."""
    words = tidy(text).split(" ")
    pieces, cur, n = [], [], 0
    for w in words:
        add = len(w) if not cur else len(w) + 1
        if cur and n + add > target:
            pieces.append(" ".join(cur)); cur, n = [w], len(w)
        else:
            cur.append(w); n += add
    if cur:
        pieces.append(" ".join(cur))
    return pieces

def agreement(labels, groups):
    """Adjusted mutual information. 0 is chance, 1 is perfect."""
    return float(adjusted_mutual_info_score(labels, groups))

def biggest_group_share(groups):
    """Share of documents in the largest group. If this is high the clustering
    failed and the agreement score means nothing. Check it every time."""
    g = np.asarray(groups)
    real = g[g != -1]
    if real.size == 0:
        return 1.0
    counts = np.bincount(real - real.min())
    return float(counts[counts > 0].max() / g.size)

print("ready")

# **4. BUILDING THE TWO VERSIONS**

This cell reads the texts, discards very short ones, and cuts each text into pieces of roughly 178 characters (never mid-word). From every text one piece is chosen at random, so that both versions contain exactly the same number of texts. The output shows the median length of both versions and an example of each.

In [ ]:
#Step 4. Build the two versions
docs = pd.read_csv(FNAME)

missing = {"text", "gold"} - set(docs.columns)
if missing:
    raise ValueError(f"{FNAME} has no {sorted(missing)} column. Wrong file.")

if "gold_name" not in docs.columns:
    docs["gold_name"] = docs.gold
    print("Older file, subjects will show as codes rather than names.\n")

docs["text"] = docs.text.map(tidy)
docs = docs.loc[docs.text.str.len() >= MIN_CHARS].reset_index(drop=True)

rng = np.random.default_rng(SEED)
short_text = []
for t in docs.text:
    pieces = [p for p in cut_into_pieces(t) if len(p) >= MIN_CHARS]
    short_text.append(pieces[rng.integers(len(pieces))] if pieces else t[:CHUNK_CHARS])
docs["short"] = short_text

subjects = docs.gold_name.values

print(f"documents        {len(docs)}")
print(f"subjects         {docs.gold_name.nunique()}")
print(f"whole, typical   {int(docs.text.str.len().median())} characters")
print(f"short, typical   {int(docs['short'].str.len().median())} characters")
print()
print("WHOLE :", docs.text.iloc[0][:280], "...")
print()
print("SHORT :", docs["short"].iloc[0])

# **5. ENCODING THE TEXT**

A language model converts each text into a vector of 768 numbers (an *embedding*), in such a way that texts with similar content receive similar vectors. We use LaBSE, a multilingual model that also copes reasonably well with early modern German. Loading the model and encoding both versions takes a few minutes on a GPU.

It is important to note that the model reads at most 256 *tokens* (words or word fragments) per text, which for these texts amounts to roughly 896 characters; anything beyond is ignored. Since some 90 per cent of the mandates are longer than that, the "whole documents" in this notebook are, strictly speaking, the opening portion of each document. The HISTMET article avoids the truncation by representing each mandate as the *mean* of the embeddings of all its 178-character pieces, so that the whole text is covered; this notebook keeps the simpler, faster variant.

In [ ]:
#Step 5. Encode the text
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("No GPU, so this takes roughly 15 minutes.")
    print("In Colab: Runtime, Change runtime type, T4 GPU.")

model = SentenceTransformer("sentence-transformers/LaBSE", device=device)
model.max_seq_length = 256

print("\nwhole documents")
vec_whole = model.encode(docs.text.tolist(), batch_size=64, show_progress_bar=True,
                         convert_to_numpy=True, normalize_embeddings=True)
print("short pieces")
vec_short = model.encode(docs["short"].tolist(), batch_size=64, show_progress_bar=True,
                         convert_to_numpy=True, normalize_embeddings=True)
print(f"\n{vec_whole.shape[1]} numbers per text")

# **6. CLUSTERING AND SCORING**

Clustering happens in two stages. UMAP first reduces the 768 numbers per text to 50, which makes the structure of the data easier to detect; HDBSCAN then groups texts that lie densely together. HDBSCAN decides the number of clusters itself and may label texts as *noise* (`-1`) when they fit no cluster; those texts are left out of the agreement score. For each version, the output reports the number of clusters, the share of texts in the largest cluster, and the AMI with the hand-assigned subjects.

The setting `SELECTION = "leaf"` makes HDBSCAN prefer many small, fine-grained clusters. With HDBSCAN's default (*excess of mass*), the clustering of short or truncated texts collapses into a single cluster holding almost all points, which makes the agreement score meaningless. The article also differs in the short-piece condition: there, *all* 85,823 pieces are clustered as separate units, rather than one piece per mandate.

In [ ]:
#Step 6. Cluster and score
import umap, hdbscan

def find_groups(vectors, n_dims=UMAP_DIM):
    reduced = umap.UMAP(n_components=n_dims, n_neighbors=UMAP_NEIGHBOURS,
                        metric="cosine", random_state=SEED).fit_transform(vectors)
    return hdbscan.HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES,
                           cluster_selection_method=SELECTION).fit_predict(reduced)

groups_whole = find_groups(vec_whole)
groups_short = find_groups(vec_short)

results = {}
for name, groups in (("Whole documents", groups_whole), ("Short pieces", groups_short)):
    keep = groups != -1
    score = agreement(subjects[keep], groups[keep])
    share = biggest_group_share(groups)
    results[name] = {"score": score, "biggest": share, "n_groups": int(len(set(groups[keep])))}
    warn = "   <- clustering failed, ignore the score" if share > 0.5 else ""
    print(f"{name:<18} groups {results[name]['n_groups']:>4}   biggest {share:>5.0%}"
          f"   agreement {score:.4f}{warn}")

# **7. RESULT**

The bar chart compares the agreement of both versions with the hand-assigned subjects. The dashed line marks the score of the Dutch HISTMET regions against the project's themes (0.105), which have the same length as the short pieces here. The figure is saved as `bern_length_bars.png`; download it via the folder icon on the left if you wish to keep it.

For reference, this is the result we obtained (your values may differ slightly):

![Expected bar chart](https://raw.githubusercontent.com/CARomein/WorkshopBern2026_HISTMET_ANNIF/master/histmet/bern_length_bars.png)

In [ ]:
#Step 7. Bar chart
import matplotlib.pyplot as plt

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK_SOFT, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"

names = list(results)
scores = [results[n]["score"] for n in names]

fig, ax = plt.subplots(figsize=(7, 4.2), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
bars = ax.bar(names, scores, width=0.5, color=[BLUE, ORANGE], zorder=3)
for b, s in zip(bars, scores):
    ax.text(b.get_x() + b.get_width()/2, s + max(scores)*0.03, f"{s:.3f}",
            ha="center", va="bottom", color=INK, fontsize=12, fontweight="600")

ax.axhline(HISMET_SUBJECT_AMI, ls="--", lw=1.2, color=INK_SOFT, zorder=2)
ax.text(0.5, HISMET_SUBJECT_AMI + max(scores) * 0.02, "our Dutch regions",
        va="bottom", ha="center", color=INK_SOFT, fontsize=9)

ax.set_ylabel("agreement with hand-assigned subject", color=INK_SOFT, fontsize=10)
ax.set_title("Same documents, same method, different amount of text",
             color=INK, fontsize=13, pad=14, loc="left")
ax.set_ylim(0, max(max(scores), HISMET_SUBJECT_AMI) * 1.3)
ax.grid(axis="y", color="#e6e5e1", lw=0.8, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#d8d7d2")
ax.tick_params(colors=INK_SOFT, length=0, labelsize=10)
plt.tight_layout()
fig.savefig("bern_length_bars.png", dpi=200, facecolor=SURFACE, bbox_inches="tight")
plt.show()

ratio = scores[0] / scores[1] if scores[1] > 0 else float("inf")
print(f"Whole documents score {ratio:.1f} times higher than the short pieces.")
print("saved bern_length_bars.png")

# **8. THE SAME RESULT AS A MAP**

UMAP is used here once more, now to reduce each text to two dimensions so that it can be plotted. Each dot is one mandate, placed so that similar texts lie close together; colour marks the three most frequent subjects, everything else is grey. The figure is saved as `bern_length_map.png`.

Our result, for reference:

![Expected map](https://raw.githubusercontent.com/CARomein/WorkshopBern2026_HISTMET_ANNIF/master/histmet/bern_length_map.png)

In [ ]:
#Step 8. Map
def flatten(vectors):
    return umap.UMAP(n_components=2, n_neighbors=UMAP_NEIGHBOURS, metric="cosine",
                     random_state=SEED).fit_transform(vectors)

xy_whole, xy_short = flatten(vec_whole), flatten(vec_short)

top3 = list(pd.Series(subjects).value_counts().head(3).index)
AQUA = "#1baf7a"
palette = {top3[0]: BLUE, top3[1]: ORANGE, top3[2]: AQUA}

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2), facecolor=SURFACE)
for ax, xy, title in ((axes[0], xy_whole, "Whole documents"),
                      (axes[1], xy_short, "Short pieces")):
    ax.set_facecolor(SURFACE)
    other = ~np.isin(subjects, top3)
    ax.scatter(xy[other, 0], xy[other, 1], s=5, c="#d3d2cd", lw=0, alpha=0.6)
    for subj in top3:
        m = subjects == subj
        ax.scatter(xy[m, 0], xy[m, 1], s=9, c=palette[subj], lw=0, alpha=0.85,
                   label=f"{subj}  (n={int(m.sum())})")
    ax.set_title(title, color=INK, fontsize=12, loc="left", pad=10)
    ax.set_xticks([]); ax.set_yticks([])
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_color("#e6e5e1")

leg = axes[0].legend(loc="upper left", fontsize=9, markerscale=1.6, labelcolor=INK_SOFT,
                     frameon=True, framealpha=0.92, facecolor=SURFACE, edgecolor="#e6e5e1")
leg.get_frame().set_linewidth(0.8)
plt.tight_layout()
fig.savefig("bern_length_map.png", dpi=200, facecolor=SURFACE, bbox_inches="tight")
plt.show()
print("saved bern_length_map.png\n")

for s in top3:
    print(f"  {s:<36} {int((subjects == s).sum()):>5}")
print(f"  {'everything else':<36} {int((~np.isin(subjects, top3)).sum()):>5}")

# **WHAT IT SHOWS**

Cut down to the length of the HISTMET regions, the Bernese mandates score about as poorly as the Dutch HISTMET material (and in our run even lower). Left whole, they score well above it. Since nothing changed except the amount of text the model saw, length appears to be doing the work. The map tells the same story more directly: on the left the subjects occupy recognisable areas (*Matters of Religion* most clearly), on the right they dissolve into a single cloud.

The article arrives at the same conclusion with its fuller design: 0.185 for whole mandates against 0.106 for pieces (0.203 against 0.035 with leaf selection, as used here). Since 0.106 is almost exactly the Dutch figure, a different territory, language and genre reproduce the HISTMET result once the unit of text is matched; hence the conclusion that genre winning over theme is a property of the unit of analysis rather than of the method or of early modern Dutch.

Three caveats deserve mention. First, the whole-text version always contains the opening of the document, where the title and the thematically most explicit formulations tend to appear, whereas the short piece is taken from a random position; length and position are therefore partly confounded. Second, a 178-character piece inherits the subject of the mandate it was cut from but may not itself express it, which exaggerates the drop in agreement. Third, an AMI of around 0.2 remains modest: the clusters capture subject only partially, even with longer texts.

**To go further:** change `CHUNK_CHARS` in Step 3 to 400 or 800 and rerun Steps 4 to 8, to see where the method begins to work on this kind of material. To test the position effect, replace `pieces[rng.integers(len(pieces))]` in Step 4 with `pieces[0]`, so that the short piece is always the opening of the text.

**Further reading:** on the vocabulary, C. Annemieke Romein, Andreas Wagner and Karl Härter, 'The 2020-Systematic, Multilingual Categorisation of Policeymaterien for Early Modern Police Ordinances', *Journal for Digital Legal History* 4, no. 1 (2025), [doi:10.21825/dlh.93163](https://doi.org/10.21825/dlh.93163); on the Bernese corpus and earlier Annif experiments, Christel Annemieke Romein, Sara F. Veldhoen and Jaap C. Romein, 'Applying (semi-)automatic metadata to early modern normative texts. Annif and Policeygesetzgebung from the City-State of Bern (1528–1798)', *Digital Scholarship in the Humanities* 41, Suppl. 1 (2026), i230–i237, [doi:10.1093/llc/fqaf072](https://doi.org/10.1093/llc/fqaf072). The HISTMET article is currently under review.

**To cite this notebook:** Kamyab Karimi and C. Annemieke Romein, *Does Text Length Explain the Clustering? Testing Unsupervised Subject Discovery on Early Modern Normative Texts from Bern (1528–1798)* (version 1.0, 2026), Zenodo, DOI: *to be added*.

**Next:** Part Two shows what happens when the hand-assigned labels are used to *train* a classifier rather than merely to evaluate clusters. Note that the scores are not directly comparable, as Part Two measures F1 rather than AMI. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CARomein/WorkshopBern2026_HISTMET_ANNIF/blob/master/02_Annif_subject_indexing.ipynb)